# 第 4 章 · 完成、限制与异常控制流

**这一章你会得到什么**：理解 mini-SWE-agent 如何用**异常**统一表达“该结束了”——包括正常提交、超限、格式错误，以及为什么 `Submitted` 能从 Environment 一路传回 `run()`。

In [8]:
# 环境自检：把源码目录加入 sys.path，并切到仓库根目录
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("Python:", sys.version.split()[0])
print("mini-SWE-agent:", minisweagent.__version__)
print("仓库根目录:", REPO)

Python: 3.13.14
mini-SWE-agent: 2.4.5
仓库根目录: /Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent


## 本章用到的类与函数（速查）

下面实验会 import 这几样东西，先混个脸熟：

| 名字 | 出处 | 一句话 |
|------|------|--------|
| `LocalEnvironment` | `environments/local.py` | 本地执行环境，真正跑 bash 的那层。常用参数：`cwd`(工作目录)、`timeout`(超时秒数) |
| `Submitted` | `exceptions.py` | "任务完成"异常。环境检测到完成信号时 raise 它，`.messages` 里带着一条 `role="exit"` 消息 |
| `DefaultAgent` | `agents/default.py` | 主 Agent 类（`run()`/`step()` 在里面）。构造要传 model、env 和配置参数 |
| `DeterministicToolcallModel` | `models/test_models.py` | 确定性"假模型"：按 `outputs` 列表顺序吐固定回复，不联网不花钱 |
| `make_toolcall_output(content, tool_calls, actions)` | `models/test_models.py` | 小工具：快速造一条"模型回复"。`actions` 是命令列表，`tool_calls` 本章传 `[]` 即可 |

## 概念：一个异常家族

所有“打断主循环”的信号都继承自 `InterruptAgentFlow`：
```text
InterruptAgentFlow
  ├── Submitted          任务完成
  ├── LimitsExceeded     成本/步数超限
  │     └── TimeExceeded 墙钟时间超限
  ├── UserInterruption   用户打断（交互式）
  └── FormatError        模型输出格式错误
```
每个异常都携带要追加进 `messages` 的消息。`run()` 只需分别 `except` 它们。

In [3]:
# 小工具：带行号打印源码切片（相当于 nl + sed）
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

In [4]:
show_source("src/minisweagent/exceptions.py", 1, 27)

 1  class InterruptAgentFlow(Exception):
 2      """Raised to interrupt the agent flow and add messages."""
 3  
 4      def __init__(self, *messages: dict):
 5          self.messages = messages
 6          super().__init__()
 7  
 8  
 9  class Submitted(InterruptAgentFlow):
10      """Raised when the agent has completed its task."""
11  
12  
13  class LimitsExceeded(InterruptAgentFlow):
14      """Raised when the agent has exceeded its cost or step limit."""
15  
16  
17  class TimeExceeded(LimitsExceeded):
18      """Raised when the agent has exceeded its wall-clock time limit."""
19  
20  
21  class UserInterruption(InterruptAgentFlow):
22      """Raised when the user interrupts the agent."""
23  
24  
25  class FormatError(InterruptAgentFlow):
26      """Raised when the LM's output is not in the expected format."""


## 运行看结果 1：完成信号如何变成 `Submitted`

`LocalEnvironment` 执行命令后会检查输出第一行是否等于魔法字符串
`COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT`。命中就抛 `Submitted`。

In [5]:
from minisweagent.environments.local import LocalEnvironment
from minisweagent.exceptions import Submitted

env = LocalEnvironment(cwd=str(REPO))
try:
    env.execute({"command": "echo COMPLETE_TASK_AND_SUBMIT_FINAL_OUTPUT\necho agent harness"})
except Submitted as e:
    print("异常类型:", type(e).__name__)
    print("携带消息:", e.messages)

异常类型: Submitted
携带消息: ({'role': 'exit', 'content': 'agent harness\n', 'extra': {'exit_status': 'Submitted', 'submission': 'agent harness\n'}},)


## 观察点

- `submission` 是**魔法行之后**的内容（`final-answer`），不是整段输出。
- 完成判断放在 **Environment**，而不是循环层：因为“完成”往往是模型执行命令后才产生的**输出**，只有真正跑过命令的一方才看得到。回想第 1 章你的玩具版把判断放在循环层——两种设计都能 work，但这种更能处理“执行后才知道结束”的情况。

## 运行看结果 2：step_limit 如何结束一个永不提交的任务

In [18]:
from minisweagent.agents.default import DefaultAgent
from minisweagent.models.test_models import DeterministicToolcallModel, make_toolcall_output

action = {"command": "echo only-one-step", "tool_call_id": "call_1"}
model = DeterministicToolcallModel(outputs=[make_toolcall_output("第一步", [], [action]), make_toolcall_output("第2步", [], [action]),make_toolcall_output("第3步", [], [action])])
agent = DefaultAgent(model, LocalEnvironment(cwd=str(REPO)),
                     system_template="system", instance_template="{{task}}",
                     step_limit=2, cost_limit=5)
result = agent.run("这个任务永远不会主动提交")
print("result =", result)

for i, m in enumerate(agent.messages):
    print(i, m.get("role"), "|", m.get("content"))

result = {'exit_status': 'LimitsExceeded', 'submission': ''}
0 system | system
1 user | 这个任务永远不会主动提交
2 assistant | 第一步
3 tool | <returncode>0</returncode>
<output>
only-one-step
</output>
4 assistant | 第2步
5 tool | <returncode>0</returncode>
<output>
only-one-step
</output>
6 exit | LimitsExceeded


## 动手：预测再验证

不运行下面这格，先在心里回答：把 `timeout=1` 的命令跑一个 `sleep 2`，会发生什么？
它会让**整个 Agent 退出**，还是只产生一条 observation？写下你的答案，再运行核对。

In [19]:
# 单个命令超时 —— 观察 returncode 和 exception_info
out = LocalEnvironment(timeout=1).execute({"command": "printf partial-output; sleep 2"})
print(out)

{'output': 'partial-output', 'returncode': -1, 'exception_info': "An error occurred while executing the command: Command 'printf partial-output; sleep 2' timed out after 1 seconds", 'extra': {'exception_type': 'TimeoutExpired', 'exception': "Command 'printf partial-output; sleep 2' timed out after 1 seconds"}}


## 观察点

- 命令超时只是**一条 observation**（`returncode` 非 0、带 `exception_info`），Agent 不会因此退出——它会把这条结果喂回模型，让模型决定下一步。
- **单个命令超时 ≠ 整个 Agent 超时**。后者是 `wall_time_limit_seconds`，会抛 `TimeExceeded`（属于 `LimitsExceeded`）。
- 这就是“能力边界 vs 控制流边界”的分水岭：Environment 报告事实，`run()` 决定是否终止。

## 闭卷检查
1. 模型、Environment、`run()` 在完成链路里各负责什么？
2. 为什么 `Submitted` 能从 `env.execute()` 一路传回 `run()`？（提示：异常传播 + `run()` 的 `except InterruptAgentFlow`）
3. 连续格式错误达到上限后会发生什么？（看 `default.py` 第 100-110 行）

**完成标准**：你能说清“谁发出完成信号、谁识别、谁让循环退出”，并区分单命令超时与 Agent 超时。